# TF‑IDF → AWS XGBoost (Multiclass, Imbalanced) — Ready‑to‑Run SageMaker Studio Notebook

This notebook trains a 5‑class text classifier using **TF‑IDF** features with the **AWS built‑in XGBoost container**.

**Highlights**
- Stratified **80/10/10** splits
- **Instance weights** (balanced) for class imbalance
- Sparse **LibSVM** format for TF‑IDF features
- SageMaker **Training** + **Batch Transform** + local evaluation (macro‑F1, per‑class F1, confusion matrix)

> Replace the **DATA LOADING** cell with your real dataset if you already have one in S3 or local.


## 0) Environment & Versions

In [ ]:
!pip install -q sagemaker==2.* scikit-learn==1.* pandas==2.* scipy==1.* xgboost==1.* boto3==1.*
import os, json, boto3, sagemaker, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.datasets import dump_svmlight_file
from scipy import sparse

sess = sagemaker.Session()
region = sess.boto_region_name
role = sagemaker.get_execution_role()
s3 = boto3.client('s3')
bucket = sess.default_bucket()  # or set your own bucket
prefix = 'poc-tfidf-xgb-multiclass'
print('Region:', region)
print('Role:', role)
print('Bucket:', bucket)


## 1) Data Loading (replace with your dataset if available)

In [ ]:
# ▶️ Replace this with your real data load.
# Expect a DataFrame `df` with columns: 'text' (str) and 'label' (categorical in {A,B,C,D,E}).
# Examples:
# df = pd.read_csv('s3://your-bucket/path/to/texts.csv')
# df = pd.read_parquet('s3://your-bucket/path/to/texts.parquet')
# assert set(df.columns) >= {'text','label'}

# For POC purposes, we synthesize an imbalanced dataset that mirrors your counts:
counts = {'A':145368, 'B':18289, 'C':6432, 'D':1783, 'E':2169}
df = pd.DataFrame({
    'label': sum([[k]*v for k,v in counts.items()], []),
    'text':  sum([[f"sample text about topic {k}"]*v for k,v in counts.items()], [])
})
print(df.shape)
df['label'].value_counts()

## 2) Stratified 80/10/10 split (no leakage)

In [ ]:
X = df['text'].values
y = df['label'].values

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42
)

def describe_split(name, yv):
    import pandas as pd
    vc = pd.Series(yv).value_counts().sort_index()
    print(f"\n{name} size={len(yv)}")
    print(vc)
    print('proportions:' , (vc/vc.sum()).round(4).to_dict())

describe_split('TRAIN', y_train)
describe_split('VAL',   y_val)
describe_split('TEST',  y_test)

# Label mapping → integers 0..K-1
classes = np.sort(pd.unique(y_train))
cls_to_id = {c:i for i,c in enumerate(classes)}
id_to_cls = {i:c for c,i in cls_to_id.items()}
y_train_i = np.array([cls_to_id[c] for c in y_train], dtype=int)
y_val_i   = np.array([cls_to_id[c] for c in y_val], dtype=int)
y_test_i  = np.array([cls_to_id[c] for c in y_test], dtype=int)
print('\nClass → id mapping:', cls_to_id)


## 3) TF‑IDF Vectorization (fit on **train only**)

In [ ]:
tfidf = TfidfVectorizer(
    stop_words='english',
    min_df=2,           # adjust for your corpus size
    max_df=0.9,
    ngram_range=(1,2),  # start with uni+bi; tune later
)
Xtr = tfidf.fit_transform(X_train)
Xva = tfidf.transform(X_val)
Xte = tfidf.transform(X_test)
Xtr.shape, Xva.shape, Xte.shape


## 4) Class‑Balanced Instance Weights (for imbalance)

In [ ]:
w = compute_class_weight(class_weight='balanced', classes=np.arange(len(classes)), y=y_train_i)
w_map = {i:wi for i,wi in enumerate(w)}
train_weights = np.array([w_map[i] for i in y_train_i], dtype=float)
print('Class weights:', w_map)
print('Weight stats: min', train_weights.min(), 'max', train_weights.max())


## 5) Save LibSVM + Weights, Upload to S3

In [ ]:
os.makedirs('data', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)

dump_svmlight_file(Xtr, y_train_i, 'data/train.libsvm', zero_based=True)
dump_svmlight_file(Xva, y_val_i,   'data/validation.libsvm', zero_based=True)
dump_svmlight_file(Xte, y_test_i,  'data/test.libsvm', zero_based=True)
np.savetxt('data/train.weights', train_weights, fmt='%.6f')

import pickle
with open('artifacts/tfidf.pkl', 'wb') as f: pickle.dump(tfidf, f)
with open('artifacts/label_map.json', 'w') as f: json.dump({'cls_to_id':cls_to_id, 'id_to_cls':id_to_cls}, f)

def up(local, key):
    s3.upload_file(local, bucket, f"{prefix}/{key}")
    return f"s3://{bucket}/{prefix}/{key}"

train_s3 = up('data/train.libsvm',       'train/train.libsvm')
val_s3   = up('data/validation.libsvm',  'validation/validation.libsvm')
test_s3  = up('data/test.libsvm',        'test/test.libsvm')
w_s3     = up('data/train.weights',      'weight/train.weights')
vec_s3   = up('artifacts/tfidf.pkl',     'artifacts/tfidf.pkl')
map_s3   = up('artifacts/label_map.json','artifacts/label_map.json')

print('Train:', train_s3)
print('Val  :', val_s3)
print('Test :', test_s3)
print('Weight:', w_s3)


## 6) Train with AWS XGBoost built‑in container

In [ ]:
from sagemaker import image_uris, inputs
image_uri = image_uris.retrieve(framework='xgboost', region=region, version='1.7-1')
xgb = sagemaker.estimator.Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    volume_size=30,
    output_path=f's3://{bucket}/{prefix}/output',
    sagemaker_session=sess
)
xgb.set_hyperparameters(
    objective='multi:softprob',
    num_class=len(classes),
    num_round=500,
    eta=0.05,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    tree_method='hist',
    seed=42,
)
xgb.fit(
    inputs={
        'train': inputs.TrainingInput(train_s3, content_type='text/libsvm'),
        'validation': inputs.TrainingInput(val_s3, content_type='text/libsvm'),
        'weight': inputs.TrainingInput(w_s3, content_type='text/csv'),
    },
    logs=True
)


## 7) Batch Transform on Test set

In [ ]:
transformer = xgb.transformer(
    instance_count=1,
    instance_type='ml.m5.large',
    assemble_with=None,
    output_path=f's3://{bucket}/{prefix}/batch'
)
transformer.transform(
    data=test_s3,
    content_type='text/libsvm',
    split_type='Line',
    join_source='Input',
    input_filter='$[0:]',
)
transformer.wait()
print('Batch transform completed.')


## 8) Download Predictions & Evaluate (macro‑F1, per‑class F1, confusion matrix)

In [ ]:
resp = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/batch")
keys = [o['Key'] for o in resp.get('Contents', []) if o['Key'].endswith('.out') or o['Key'].endswith('.csv')]
if not keys:  # try to fetch any object under batch output
    keys = [o['Key'] for o in resp.get('Contents', []) if o['Key'].startswith(f"{prefix}/batch")]
pred_key = sorted(keys)[-1]
local_pred = 'predictions.out'
s3.download_file(bucket, pred_key, local_pred)
print('Downloaded:', pred_key)

# Parse probabilities (last K columns per line)
probs = []
with open(local_pred) as f:
    for line in f:
        parts = line.strip().split()
        p = list(map(float, parts[-len(classes):]))
        probs.append(p)
probs = np.array(probs)
y_pred_i = probs.argmax(axis=1)
y_pred = np.array([id_to_cls[i] for i in y_pred_i])

print(classification_report(y_test, y_pred, digits=3))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred, labels=list(classes)))


## 9) Utility: Single‑text Inference Helper (local TF‑IDF + hosted model)

In [ ]:
# Optional: deploy for real‑time inference
# WARNING: This creates an endpoint — remember to delete when done to avoid charges.
deploy_endpoint = False  # set True to deploy
if deploy_endpoint:
    predictor = xgb.deploy(
        initial_instance_count=1,
        instance_type='ml.m5.large',
        serializer=sagemaker.serializers.CSVSerializer(),
        deserializer=sagemaker.deserializers.CSVDeserializer(),
    )
    endpoint_name = predictor.endpoint_name
    print('Endpoint:', endpoint_name)
else:
    predictor = None
    endpoint_name = None

def predict_texts(texts, tfidf_path='artifacts/tfidf.pkl', label_map_path='artifacts/label_map.json', endpoint=None):
    import pickle, json
    with open(tfidf_path, 'rb') as f:
        vec = pickle.load(f)
    with open(label_map_path) as f:
        lm = json.load(f)
    id_to_cls = {int(k):v for k,v in lm['id_to_cls'].items()} if isinstance(list(lm['id_to_cls'].keys())[0], str) else lm['id_to_cls']
    Xs = vec.transform(texts)
    # For real‑time endpoint, we need dense CSV or LibSVM; we'll use dense CSV for simplicity
    from io import StringIO
    import numpy as np
    arr = Xs.toarray()  # beware memory on huge vectors; OK for small batches
    csv = '\n'.join([','.join(map(str, row)) for row in arr])
    if endpoint is None:
        print('No endpoint provided (set deploy_endpoint=True to create one).')
        return None
    pred = predictor.predict(csv)
    # pred is a list of lists of probabilities
    yhat = np.array(pred).argmax(axis=1)
    return [id_to_cls[int(i)] for i in yhat]

print('Helper ready. Deploy endpoint to use predict_texts([...], endpoint=predictor).')


## 10) Cleanup (optional)
Delete the endpoint if you created one.

In [ ]:
if 'predictor' in globals() and predictor is not None:
    predictor.delete_endpoint(delete_endpoint_config=True)
    print('Endpoint deleted.')
else:
    print('No endpoint to delete.')
